# Profile the current Reversability training path

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
This notebook compares baseline batch 29, midpoint batch 29 and midpoint batch 33 using the frozen study model and data. It runs short independent probes, not full training or checkpoint resumes. New evidence goes into a unique Drive folder.

The tested profiling harness is embedded, so no GitHub update or ZIP upload is needed. The original implementation is fetched at commit `d071e1e2d8e4575d2fc9ca5ccb1ef4e3d5e185b7` and checked against its recorded source hashes.

Timing, operator traces and allocator diagnostics run in separate processes. Only the uninstrumented timing pass supplies throughput claims. The default is 5 warmup updates, 20 timed updates and 1 traced update per scenario.


In [ ]:
import importlib.metadata, subprocess, sys
# Match the recorded Torch/CUDA build and NumPy. Python differences are recorded below.
required = {"torch": "2.11.0+cu128", "numpy": "2.1.3"}
wrong = []
for name, version in required.items():
    try:
        if importlib.metadata.version(name) != version: wrong.append(name)
    except importlib.metadata.PackageNotFoundError:
        wrong.append(name)
if wrong:
    if "torch" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cu128",
                        "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
    if "numpy" in wrong:
        subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3"], check=True)
    imported = [name for name in wrong if name in sys.modules]
    if imported:
        raise RuntimeError("Packages changed after import. Choose Runtime → Restart session, then run all cells again.")
print("Pinned packages installed. If Colab requests a restart, restart the session and rerun these cells.")


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, uuid, platform, torch, numpy as np

drive.mount("/content/drive")
if not torch.cuda.is_available():
    raise RuntimeError("Select a T4 GPU runtime before continuing.")
if torch.cuda.get_device_name(0) != "Tesla T4":
    raise RuntimeError("Select T4 for the current bottleneck comparison. Another GPU would be a separately declared experiment.")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
DRIVE_ROOT = Path("/content/drive/MyDrive/Reversability")
OUTPUT = DRIVE_ROOT / "profiling-v1" / RUN_ID
REPO = Path("/content/reversability-profile-" + RUN_ID)
PINNED_COMMIT = "d071e1e2d8e4575d2fc9ca5ccb1ef4e3d5e185b7"
subprocess.run(["git", "clone", "https://github.com/LokeshJatangi/Reversability.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT], cwd=REPO, check=True)
manifest = json.loads((REPO / "results/recorded/reversible-v1/source-manifest.json").read_text())
for relative, expected in manifest.items():
    if hashlib.sha256((REPO / relative).read_bytes()).hexdigest() != expected:
        raise RuntimeError("Frozen source differs: " + relative)
reference_config = REPO / "results/recorded/configs/baseline_colab.json"
reference_metrics = REPO / "results/recorded/runs/baseline_20m_fineweb_edu_50m_v1/metrics.jsonl"
startup = next(json.loads(line) for line in reference_metrics.read_text().splitlines()
               if json.loads(line).get("event") == "startup")
if hashlib.sha256(reference_config.read_bytes()).hexdigest() != startup["config_sha256"]:
    raise RuntimeError("Recorded baseline configuration checksum differs")
actual = {"torch": torch.__version__, "numpy": np.__version__, "cuda_runtime": torch.version.cuda,
          "python": platform.python_version(), "gpu": torch.cuda.get_device_name(0)}
expected = {"torch": startup["environment"]["torch"], "numpy": startup["environment"]["numpy"],
            "cuda_runtime": startup["environment"]["cuda_runtime"],
            "python": startup["environment"]["python"].split()[0],
            "gpu": startup["environment"]["accelerator"]["name"]}
mismatches = {key: {"recorded": expected[key], "actual": actual[key]}
              for key in actual if actual[key] != expected[key]}
if any(key != "python" for key in mismatches):
    raise RuntimeError("Runtime differs from the intended comparison: " + json.dumps(mismatches))
# Python patch/version differences are declared; these are new paired probes, not a rerun claim.
print("Runtime:", actual)
print("Declared differences:", mismatches)
print("New evidence destination:", OUTPUT)


## Stage the original dataset

The default points to the original Drive artifact root. The existing staging tool also checks restored `baseline-recovery-*` folders and recovery ZIPs. If it reports multiple candidates, set `ARTIFACTS` to the intended artifact root and rerun this cell. It verifies the original manifest, train/validation binaries and tokenizer hashes, and copies data to local runtime storage. It never downloads or prepares replacement training data.


In [ ]:
ARTIFACTS = DRIVE_ROOT / "artifacts"  # Change only if staging reports ambiguous candidates.
LOCAL_DATA = Path("/content/reversibility-profile-data") / "data_fineweb_edu_gpt2_50m_v1"
sys.path.insert(0, str(REPO / "scripts"))
from stage_baseline_data import stage_data
receipt = stage_data(ARTIFACTS, LOCAL_DATA, DRIVE_ROOT)
if receipt["manifest_sha256"] != "71a12f5765cf37b7f5fb8753d51fa0828915a6db313ae20aef7fa9fb2e14df8a":
    raise RuntimeError("Dataset is not the frozen study stream")
print(json.dumps(receipt, indent=2))


In [ ]:
import base64, zlib
# Embedded tested harness; only this new file is written into the isolated checkout.
PROFILE_BYTES = zlib.decompress(base64.b64decode('eNrdPGlz3DaW3/UrEOaD2UqLumI7lqenxknsVKrWtsrHTtVqVSw2iVYz4jUEKanj0X/fdwAgeHRLTjJf1lWJmgTw8PDwbjzw228OW1UfLtPiUBY3oto067I43fM876OsojpqpGjSPC2u5qKsJDyXtWjqKJZKREUioiwrY3qZpNFVUaomjdVLUcgbWYtaqjaX2D0tAEIAQPdWdZmLMFy1TVvLMBRpXpV1A7CKsomatCzU3p55V18BBkqa57gsGnnXZOmSoSSAHOAmDQzzPEeM5e9lYUeu2iJuyjJT5sU6UmuEox9/U2VhfudRs2b4FfyCTgb8OTbo3zWsvczNE8CSd/YBl0FksG/aZVWXQLHuzcb+RFTtios2rzYiUqKobHtZx+veQ1AUAS0IiBVl2PvN3t6H9+8/iQXh6AN10wxoOwtgA8rsRvqzAAgpi0ZdHF/uweQBLi1ICyXrxj+aA861TxAOhafq2JvNmAI1bqNKl2mWNhtDh2WbZkmYl4nMLDncfoF5yiQi1z0xTOKGcBkpmaWF3btl1MTrMGrmgnBX6+jk6bO5qNsCCRTmsolge6O9vTcf3v/P63fh21fvfn3z+iOu2Xt+HB2frJ4+f/Y0Xp0+Xz5fPV0tf3j+9DR5eryKjn44+eHF8dPoWbI8PT6N5MlRJFfPV9GL1fJEHn+frH6IvL29vUSuhIqAVEiZubiJslbOzvYE/COa4utZcFunjQyRCX1kmSCB/VI+dZ6LtEiAxIuTOQnFbVhExeJNlCk5E98J738Lb2bm2RSxn8ibNDZTpCvBz0GzqaRYwKLiNok8bsV/vPP4MsDh67os0t+lgQKA4yxSSnyQVyhCPA7E7Z1UjUxEJaNrBXxY36Q3UsRr2EAUTtmAqDrgEleIQfykAKEUSwkd8qi+ViTACBlXEYYg1E0Y+rCRq7nGfy6QL2Yd3tgYmDZ6wA6wa27/fu+6vFW6bwRMfoO9Ly7n8F+/Hy4q1PoHUF84b3Ft9Q29PNqzGKsorzJJ+DoYAu0dJHkDvhlvAP6rJSitwr5y53b2J4/ugF/zst502PnOFLO9Dp5Fc3q86TA9fJoKMNyfaJl36E5BcFDpAzANc4ttN3wFah92Czjf3a4BzcrbC6+PSrjcNFJ5l3quHT0mke5DNVhtBTrs4C6EYP6jsyuB/plHRXQla8s5NYmV5vQiyuUE/zBfA+OwuRywziaVWbKLmcZg0MgOoNymzVqzClgU1JQ1aNu4rJPQWATf0y2HHqidAa6PxoZU1Ham05LkcDJwwUJ88ZhQ3hlNPBfT23omjkzTYHOg5X6nYC62CCbuNhjeupnkMle4HiWYA5gTPDYGuVtWBzqclG9IJNDD0W1QO0jOshVEVSWLBNm7a2/qzS5mW6XgJmSDLts3ePsmD3GpymrQ+hVbZkkMy/nrNq0H9S/atj+zdT2j1tu8vb0degd1juMR++RszbUaUjNr3j9J9J6ieiNuawReK1EWsNWi0ssSzVqKsk6vkAcE+F+/y4JMbna4jOLr26hOwK1OpLXsLfi1ZG+t7kPIfnlbyHqupTqLljJztIqFvxBX4Kc14Ew6/TtiwMsQW9Nl26BRx1a0HdQ5CMMkjcGfsN3/Yb32AHFQvpmng+jQCXZwHwIG8Bz293FdV2qg94Bp2ygLCXmYnP76M2TZGBgrAv/U54UJCT4bt4+Vr96CQFsEF+iEnmWtagk0wtAR8RHd5t3K+guGrQV2BME8G8pej8BjbCYnGRMV/yEJxgCAbbbvLnKO5XBGkRt76gldhiWI8TVuPDFiQI9qYOiQ6aghgBkLMBcwELnVg59xCVxfgK99CG3wBzYiNM2zbWDyrNoGBZomxtNYxrApr2URynwpk4Qi4Wk4g27TsKpSpYTxg+DGPachkn4Pi7LOt0HqekxDyPJwLaNk2/CybaoWVF5d/ibJx3CgANcZmddKyjOKhUJEgPQOoMzIDKCrUkCs2caNN7HfXaQIUNICH2RYRWmN6HSNh6bJAddHahqiXpuBmEX5MonOepDRvmQEN4cVS28kAbhetgWpCq/qKAllgaoj8bXacKGBXyZ3MSYDitqmREiAIf4ZrJUxuvmtcgB05n1k2skjZ+FDAWNAgB0K52wYzGg1oqPSskLH9XdZh22FmRSzn/Y9xGSAC/6lEH6O+ZhVCuwbt7Uq6y4E7JspiuxB4XJvcMrXG5XiojjkVwDaY2uTyeKqcbsq+a8Wgk8ZcovuFsVxm7cZmUanMxIvBXYN3XawzbJSemBctkUDIxijfTPffg8iT5FX0M83LNJtVBypxmd3BtaL/s3ii7eqjp95ptcqK6Pm+Nm8R2tvuXK6LHWf+wtLE2C5VCETX17OZlvTAZRxmxhD8eqqOj3xmAkdx6Jos0w/7m0zYh5v+CHscUlM7bnG3Wx/YJt9dICaEhRKIRef6lbr+axUCl435AgcBUf0Uie+dChuWDQk5oyKK+m7tHem3YUnJoJCIEUIfFiolay9AWvfzcWGcjhpYrYb7KJv2JbYFYIji9uc+8w1RzhdiGVmI+AA9Q6sgkm/wBvnqb8GYKWhpd61NhK6TmmMbXBWXqWNwigXe/p3s0eDjmvcIJCQuqw206AVAn4T9Hr6PCMwNllbTBce04rNi9l8rCan/6VXBapETJTdLQ6Oj45QVySsvheeanNv7Hmj1YKN/J04iDA85F159LqNPZpaMgYspNREqshQnU2upUMiMND8MarTThN5XjRHQH/8DthsCtq34i3pJ/TciWNl/UQRhKgWIMZ1RPQ6zNIVZ7+XcoXqHvsX0C7yFPcP9F7wKGkisWXwQwo5Ev0dEx8CnCaK1/6M9Rz8TRuZO9h38g4jSAIfVDttQXQhiY6ztPIGyRW7P5iMHO+Rpq2B4luF1SGFFLfBX1EEbZNCUIFzsQ3H9tD4Z1ENbm0DZhMYu2dcGLnL2YMLUvEauFqvyKLjLguodAX2F6XNH+gaYm+eFBOBnC1vIFqQjbFjNH9U5y0aqTwt/ONgvgsO9+2AdISJwcUspGNEARoY26jG45oQD3/AuHwngqdgJH3TZ9h+sGv0DEceAww8VoEwF2JZ/FGl8N5QYdYhlNV6Sdsm29crn2vc+8nIq7oEomBIaa0W7WdIDYMYg97BDDVlBrJ675FKoYONQubw/lj+tWguDJMC/UNWArMpHiZ4E/zrsjn7ZuO0y2gC8Tc9/URYGqXgJbxBCQbSnpdp0byu67L2vVdvz0UJPiNI9y1mETCTCF0ETwu0B010nUKQl7wEqQKdAzS8WoOvLOIsSo3+1mHvFw9VBrg9jh7Ryls7u+SvUVxyRkJq1Qu0G3Y963TKvXZVQb7ia+Om4iFj1nQHKagliMdStcITCtPjgrEBjgRGoXONssx848wPuvaRA5+sI+J24r1DnkUwtOBDAwP5saMk/FpKexTUyyDtOhKaOJGwVN6SZEXO3Zpnxcb7vceAeehkw/hAfYd3etYHTjk0KLPNELzADtTlrU+/NGEsvph+AJjUFlzLDcWswFv2HXNaH6+4apkTw1Z1HfElHjNSC8DBlF2Ib51O9p3Tsw+b0XfBm9CYIMy7HhYCRBFHwZB2NJPuOgnG7fAVsDS5wcqB/2y2ZBdsPaBV0RUG8kdD4GmBeQG1jirZ0cl9aXbytqyvZe07WTnW76AZ6RwVhClR1BxwA5hUMLh00Kqtg35vYlPZrMtkQSP491xMBJbcgZ37CceMAokkrRd4/k1dzRsy/XmVZlIf4RqZdGakuCxPkwpVgCOXfUx72ZBFLwWC5+6yYkSDp/0pkNFUXl7LIdx+4GfiPp2txV2zjg4/+sjy3hiuTleQTpntPISmqBMVpiO9qQqjmyjltOlIOX7gU3utFn/6/PMr8cv5Z5Dcf7UpOPyBODgAPA4YD7CxVdbmS1SO5DByYI957EBjRphzFirIr2F7fF3OQNHnHLziVDVhee3uFWtyOpRC7Upy5NVtgc4E5lgoOXWm9xN+c8Yrh7XCWyroCH6DjfWxWgI63YyiHDznkwA6r8K2iWGQKT8JCtBYpgIFfM0YnGRVrtDrb8bBksfbCuNNOkW/CZ1CCGh1nlxBGcOTxU1al0UuSRMOCyi6kNWzrBAirQ0t7Fu2DlQT4dAfK0SYtAFKrjfXhJ5I9WJmfYJlkZPAIvqcNpnr3MhcM/IwSUX89N9YYKG56XOh2gpLRrCswYKdTc06Sh+Jv3euKjr73iU72aOODyHxE2dUgPFiKRMlyEtFhHTKVtiUrU4p9BFEEgykcVtmB6WQCLRFCrEttBTxHyTfj2+On4nWoSEg26xBBkFAHSRBDiC8BKHCuhcQop5yRB4wHZgLRuP6ir0HbazaO3G9sHAN46OKc1m/B+oxRN0OGFyqQTHRQ8R7h+SHMFufo6mmTTZkRJRs7NIdKm4jHgV1AWg8z10AEQd1FUTViCeYBcTRLvnCM4V0XSgHgdUxhlgn6NS6dCLy4PBuNDYrZHkCgz8MKR5a9yc9rwCU8bARUTvEEjoI1VSO1tVzT6OhM57uVXjEmkeVLqnibCm8bcFcHj/jyh8wiM5QStYagrH9VJ3RHdRXsAGjJiqVCLcPoEoK7n/c4ck0DCub8TYzm/zwdj2y/xVZZ3NqMJ4OImOaT8fy3/H6Z6ClMALete3zXTmCB3XAOYYg4JO1WaI1GHG1NeiGv828Bndnp7j+EXxi0DkdmWQCs7/End/ZzhoM1odHp1NdXLo9opSgF1VYoBiibAVM6h9zw10to+47czK5AVHAH2upLknEBUf9zFFIgcfEMFW2dSxd1fYFPU8gFx6spjfoyFMdJsSbfWmeUXoDUxu9lV/Yos24TqtGHeran9CW3FYbMK3DXv0STO6zr9iCuHWgh726Tm8WXGXl0vf2ccRsdnk/PiSwzie9CV4lUf7PycRaVi+2pHimHPUleC9qYXcTH49dKcAXJ7C/c3Er06t1AwFMHG3sDO5Llwt0lsfgDK5c8AvI80d6rY95/sBpzIK8crTZpHIw1O5MQsbaURdq+ubMzGg1JzfH+TxzbPKHxGFbrcrwpGJ4JtNppYEy6WVe/roTQ6LKrJ/UMvnMxYTq7NepTZRVkrjxCljOdG5WGTfXZBGdNJNBan9/ilb3//ldYLxVV37Tizl3lhcmbc2lQv3BE3vLNmaiOMSh5EtBdW8oGBjPAN1XrNNAKsbHHngCAF3/am54AEO7YlNzMoGqOOCFjGH1+BgXABA1+Q08fGtzhmMAj+FOlxn1trnMCGYKSJAg79nVIEdS/qhrU23u2/aJwzYvl1Hh9rcXEAJsccdi3yTd3hvbds+lGtiB6eHU5IxG1gWPyX3zd3HMOvEomIDtUpJnoAnQJ9yf8p8OH0Ua1dbgDFBmIeyEfTvYhxUAWM1pF3NUcJbeQDgoSSgHVbvn+scr7rQJfjr/fDlM6j+yjLKbyMrCQ5N9/vnVoNqV7dGkZaK1TZwz76hQnA+rlPUPv8N10f2cPkDWhc2cSdSZHuPf8Abpl4gM7lV8zdUIeOtEdxwT65Hq8D+i2Ax1Jmf70yrpq9TSVuOpjyeReIG8wyRBiNdDcvIpYyRYPcwHUQMnAgY2nDLCxP5OLp99Wdkdi0jM3ocRuJrRlVQ+ncuFy03oZJJ5Y/sCMpWaMlfUlMlOMQazSZWst4N08oXssNJYgwDKCz5luAzIjijkNKfAfogQEguWglnZLSn67c4kp2MnDhn6RWjbF9zcNV7vhtJXEjdoKJlLi1huFmYxc6ynDjPg9Gbx9Gg2TUkOwNeRCnExIROQ6BoVG18GhhB6yf3Ct+Bnav0EjaSYxuzB8PzZ7FEK0mbKHsBu27noRPb6WtYFbBBEF/pOVBxVeH8xeWlwrA9/+nz+6VfMZreFjaS9aXKldAKAGVqwTKzKOI7c5R2OT5d36GwNdDg9KlQM6BeCMuV9h3V4i+DrjNAwECcEglArcb2ydYo1hxsfsxZYXoRm4PgI/812mADGGJl01OkhO/T/QrH3HV9TdDJ5PUPvmaXbo/cJbzKGqogqtS6bKSVvL/raXkGVxtfI430UJq+b/GG8pvlHF90uqLR4UsTs0XKn4m21Tnmrdo3p7kBSzKZd9i2n4wbo4BrebjVPJ/OPLZp76Ey9h4G9xvcwAveDuNMcWPJp2YLqvzPZ4MEvMiZIqmbaubBetQ6qF+RF9VfUc7gdL2Qx5XHL1UrSnSJzdqszrmoxmZDdmXTtowGaQS68N4D6mksv5li3IvHKLo06JOmkI1zx6+H7l8ZvotuvytyAp4vLhiCJWLVZhidrKgCtBLYFTSSPSIs4axXeWUUjhJU0WVQF4i1xlt4qfRsXu3NNTEb9mXegfc4WxpTfH+Bti4SdfD4apZ0PvHHCgys1wfoXoJ0035KFBza5cI7Ct2e25+MDshtAa6nLtAfEpSyXZcSvzp1PToGHGcMaBnDB2FMJ8Yzwr5/we5YFzIVXjXhNf5DQEEdI9ALOHhCVFZh7iZX71BvkpKp9+umoxq87SLXeyB84hK1qYGb3frpzAm4z0fwC9ttjkBQS95T+bOKQ/B5eroDB17rsW99nb7F0yinu+CPn9SoGpV6nJUULvmeS0+HJCzoj1o/w++QFIOHbwgvdwdZhjDucng46nJ5qx31HscVWdIbYDJAZ4jJAZYiJRoStnJO8QxeY72eZ6hbmc7xpbTDr3/qgVASdq+tMk43cHXdwkDSUYOQKc4+iz190N++wfzufi1CoRgIxxcoIeSfjlgIHmOWg9fhTEtu+QoEsdXCgCx7mHcNtKWUQ2BvPTQ8SurEzVaUDPTQDc7uzJm5l+nmWkFNzEG01APqth/L9JV0dcnCg88kOIjpHPgnUJJyd7voVQ6fNmerVO82cjZJC2zh2sEPGtfScWptBPODQSn8fZLe0dlc9ukQBqpqVd27LG78g49yfiS+kLoS/MpYXKzhnXl99jDx5f4wSMOHKY2gB2DcIcSHkLXzv1qM8D7yaIgF9YqL77koAxtrXhAETd5ss8JQLaZ4A9ywACP0Gnb1wBn389PP7z5+Qf7HiMB6KgnZN9SG/K0wD9Twxql8f4UDqVUd0LjO30U4oc9XMLWyyFsgju4Ov3pW67E4r/ZegHVQFbhbSzLufQMqGIwBZK5nuJr82BVoYOpJQ4ZJ+h9OQkUFo97MHszVkOIwJ21Kn9UCV0pxvFdaJshOr+2kmFeILd7h4wjM9uURGdTbufoI/jdeHGjLL/PrCXTud9R1xncmFIcBlZ17Z7zaONN/O43t5hOhs76sp0wGjQieNGyePQGCaNMpCZAZwz/8c9dC+kUHyvhWm/uOAeN3WMuME+N9P558FF9zgJFQYTlkTUOvkDWN0sr0CkbqSa65e2pKDA4SkPzyl0DHLWhDpzm3ns9ee6x4YdP4tPmruFf/Guoc0MU79oYI3b+mUw1S065MMeP+xAcnbOB91+SX9sXttAyt6S9McHBzgf2fD/3mdCXc3u1cYV7Mzqq/umYoWHDHJRt+4bDQ4kS0LYF8nNYLHdj0o5uTJ9rhKl3QBHi83irsz1K93wHQn+/unR2fB6eqeNuvOvXKj98pNbXV3NpBTjOpYAfm/1CBhegtAxoCIX5qLJ+PTnSeXZ8HxyrT3T6aw7XtuA3R9aJ8KwJ9czkZd+gEy9/Bcnv4OmRp3kO5ZwZaSyT84eaG5ENZdZvTds7XUXgNJGiwHVbb4kTyx01PRQlDcha7aQ3v2DBx3GFnDcGCzpz8QqwIpoy6cdK9NkOujj7AsC3/7rQ4scSw7b8BJjayrWnKmy/R8b77JhpnjA10BTNfC6qiAKNNmLsG8gnoREVclFvxdqC5cNVkEseNyvS54mLg2z4ASyqHjM34Mi24e6xI8vk/NkotfW1BIRf35CYNhLXMsjbhJ+cthacHfruh9c+6lmYBPb5RISp3ppQ9egCIA4MIifvj2v86FvYZBB8SGbg9eJ8esR+8yOC3FXsfu5QdAVeH3sYCgSGbA3BR9B+JXS2EqFzYpAXZ3LC3Ncmi7W9Xo727h+WaO9dKc3zWcwPkDQ3JcMUf8SmcSsEgx4vW+NBQCSEjPHHVMTEPwoUsqsMIFU4QK11LpM+xgW7jpaq2TTQ5D0SbVciVrDLoDUJew9EMK1p1LL8i1AIWvL2psu1GKXBNnCmJ9niEGGVoiN+BtUyxXrQWaN1xkhPW/N2nZKsB8lTYNVSiiEAbilf0soclWMnF7Hzur1hDHHRiisuzjtnYJQE7F6CW730HT9PlVe1SjLySAk+neoZ3ra7sHRBgQogj8uEN4d6DKVZNHd2LZgh6p1dyyjvjl9du3yspPx5jmUlm8LvEG3BV0MRlrViLiFeh20LW4z3HLZNO3hjXlowxVelJKVvCANrBN2sCbHD0hqbTq4ywf7YRZ8sr7iK6JYNME7tMTY5eejD2SJwOPpHNB2V/loBVwU/PuizWqjdH7XrWZJqt6ch+AEeKsm61JBBdRlOYTkz1fAC+pAeB0lYLoaLM39K8+vD5//+FTkCf9IzP8Sh+X/ZOp6D7c59QYmyXuvvTwscxdAagizKDqZb+0i01szQ8rlytFlxcZTYjEgFR8jMOJFvoaZohLMYXeFQfu1BC8qq9IhM7xiUJhqvfDy9VhmJQxxOI8KAArgGCot++G5FSje071uua2huMLjwc60fnXDrVh+9cOJN8YxkXm3jh+xiIE8TDx7XiITQKA0KSYVr5wczhdRuYSj3RWEezIouuwDajJGhD+KV6eMmNPXmxFhFMKHRrbkzUuLr3iwDFUm5gYo/J025gu8TCB/9G2UcOkxXjs8VYes1mIya2bi7XMqoX3KS02AiOKiUtB5ruyeJkINH+CxsNcEQJBqAJXQPh1YC+aRd3VMp2O0j1sZf7CeRj10kK5sD8HPUA7YNV2ZCsLI5PuwZ+9nA6+4DST+Js4dm6FB5KVxz81CB2jUD0bqwj2MclFAKvI101uDA0m79RtvVCnEX7EUcDO45ZtazBXZOgjwe4XW7Ygb+/t4B1m39O373qX5jvQ1JY4+lWHKppVZNIBjQb3QfzRYYCu/6brQc++JwdpxwWiZ99PomRDYK3C0N9vkJXNkXFh7o6gFaOYl2ft3ZiOQNGn+GlVTLqEIaUPwhD94jD0utw68vqQ050kvA216R2bYvcG5t7/AUD10jY='))
if hashlib.sha256(PROFILE_BYTES).hexdigest() != '2251d956655aafbf86845e89ca543ef7d000ca09fb8236125c6da081c0fb4aba':
    raise RuntimeError("Embedded profiling harness checksum mismatch")
PROFILE_SCRIPT = REPO / "scripts/profile_training.py"
PROFILE_SCRIPT.write_bytes(PROFILE_BYTES)
# Receipts live alongside the suite but outside its exclusive output directory.
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
RECEIPTS = OUTPUT.parent / (RUN_ID + "-receipts.json")
RECEIPTS.write_text(json.dumps({"source_commit": PINNED_COMMIT, "profile_sha256": '2251d956655aafbf86845e89ca543ef7d000ca09fb8236125c6da081c0fb4aba',
    "runtime": actual, "declared_environment_differences": mismatches, "data": receipt,
    "probe_only": True}, indent=2) + "\n")
print("Harness staged:", PROFILE_SCRIPT)


## Run the three profiling scenarios

Each scenario has three fresh processes: uninstrumented timing; a shape/stack/operator trace; synchronized phase-memory diagnostics plus an allocator snapshot. The suite continues after a failed diagnostic and preserves every successful result. A trace can consume more memory than normal training because the profiler retains tensor references; that does not revise the original batch-capacity result.

This cell uses **T4 FP16**, physical batches **29/29/33**, accumulation **2**, and context **512**. It never runs the 50M trainer, evaluates the model or writes training checkpoints. Results are probes with fresh weights and a short warmup, not measurements of converged model quality or historical training cost.


In [ ]:
WARMUP_UPDATES = 5
TIMED_UPDATES = 20
TRACE_UPDATES = 1
command = [sys.executable, "-u", str(PROFILE_SCRIPT), "--suite", "--config", str(reference_config),
           "--data-dir", str(LOCAL_DATA), "--output", str(OUTPUT),
           "--warmup", str(WARMUP_UPDATES), "--updates", str(TIMED_UPDATES),
           "--trace-updates", str(TRACE_UPDATES)]
with (OUTPUT.parent / (RUN_ID + "-orchestration.log")).open("w") as log:
    process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line); log.flush()
    returncode = process.wait()
print("Exit code:", returncode)
print("Evidence:", OUTPUT)
if returncode:
    print("One or more passes failed. Continue to the next cell to inspect preserved logs and results.")


In [ ]:
from IPython.display import display, Markdown
suite = json.loads((OUTPUT / "suite.json").read_text())
display(Markdown((OUTPUT / "REPORT.md").read_text()))
for record in suite["records"]:
    scenario, mode, result = record["scenario"], record["mode"], record["result"]
    if result["status"] != "complete":
        print("FAILED:", scenario, mode, result.get("error"))
        log_path = OUTPUT / scenario / (mode + ".log")
        print(log_path.read_text()[-4000:])
    elif mode == "trace":
        display(Markdown("### " + scenario + " — inclusive CUDA region times"))
        rows = sorted(result["regions"], key=lambda row: row["device_total_us"], reverse=True)
        for row in rows[:18]:
            print(f"{row['name']:55s} {row['device_total_us']/1000:10.3f} ms  calls={row['count']}")
        print("Ranges overlap; do not add them. See operators.txt for operator self-device time.")
    elif mode == "memory":
        grouped = {}
        for row in result["memory_regions"]:
            group = grouped.setdefault(row["region"], {"peak": 0, "rise": 0, "count": 0})
            group["peak"] = max(group["peak"], row["peak_allocated_bytes"])
            group["rise"] = max(group["rise"], row["peak_allocated_bytes"] - row["start_allocated_bytes"])
            group["count"] += 1
        display(Markdown("### " + scenario + " — diagnostic live memory"))
        for name, group in sorted(grouped.items(), key=lambda item: item[1]["peak"], reverse=True)[:18]:
            print(f"{name:55s} peak={group['peak']/2**30:7.3f} GiB  rise={group['rise']/2**30:7.3f} GiB  calls={group['count']}")
        print("Peaks include pre-existing tensors; rise is not exclusive component-owned memory.")
print("Complete status:", suite["status"])


## Export for diagnosis

The ZIP contains timings, all phase records, operator tables, Chrome/Perfetto traces, allocator snapshots, logs and provenance receipts. Keep failed-pass logs too. Open `trace.json` in [Perfetto](https://ui.perfetto.dev/) and `allocator_snapshot.pickle` in the [PyTorch memory viewer](https://pytorch.org/memory_viz) when deeper attribution is needed.

The next decision should use measured output/loss peaks and casts versus attention/MLP/reconstruction kernel time. Larger logits or a larger batch alone do not establish a savings claim.


In [ ]:
import shutil
BUNDLE = OUTPUT.parent / (RUN_ID + "-profiling.zip")
if BUNDLE.exists():
    raise RuntimeError("Export already exists; existing evidence was preserved")
shutil.make_archive(str(BUNDLE.with_suffix("")), "zip", root_dir=OUTPUT.parent, base_dir=OUTPUT.name)
import zipfile
with zipfile.ZipFile(BUNDLE, "a", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(RECEIPTS, arcname=RECEIPTS.name)
    archive.write(OUTPUT.parent / (RUN_ID + "-orchestration.log"), arcname=RUN_ID + "-orchestration.log")
print("Saved:", BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
